In [1]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image


In [2]:
import torch
from torchvision.models.resnet import ResNet
from torch.serialization import add_safe_globals

# Register ResNet and ResNet50WithDropout as safe globals
add_safe_globals([ResNet])

# Ensure you match the model's original architecture
class ResNet50WithDropout(torch.nn.Module):
    def __init__(self, pretrained=True, num_classes=2, dropout_rate=0.5):
        super(ResNet50WithDropout, self).__init__()
        self.model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=pretrained)
        num_features = self.model.fc.in_features
        self.model.fc = torch.nn.Sequential(
            torch.nn.Dropout(dropout_rate),
            torch.nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Add ResNet50WithDropout if necessary
add_safe_globals([ResNet50WithDropout])

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Explicitly set weights_only to False and map_location
model_path = "C:\\Users\\ezeki\\OneDrive\\Desktop\\MTechProject\\Models\\best_resnet50_model.pth"
loaded_model = torch.load(model_path, map_location=device, weights_only=False)
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully on device:", device)


Model loaded successfully on device: cpu


In [3]:
from torchvision import transforms
from PIL import Image

# Preprocessing for the input image
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load an image
image_path = "C:\\Users\\ezeki\\OneDrive\\Desktop\\MTechProject\\eyepac-light-v2-512-jpg\\test\\NRG\\EyePACS-TRAIN-NRG-3013.jpg"
image = Image.open(image_path)

# Apply preprocessing
input_tensor = preprocess(image).unsqueeze(0)

In [4]:
# Print the architecture of the loaded model
print("Loaded Model Architecture:")
print(loaded_model)

Loaded Model Architecture:
ResNet50WithDropout(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
       

In [5]:
from torchsummary import summary

# Ensure the model is on the appropriate device
loaded_model = loaded_model.to(device)

# Print a detailed summary of the model
print("Model Summary:")
summary(loaded_model, input_size=(3, 224, 224))  


Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14    

In [6]:
# Define the class labels
class_labels = {0: "NRG", 1: "RG"}

# Perform inference
with torch.no_grad():
    output = loaded_model(input_tensor)

# Get the predicted class
_, predicted_class = torch.max(output, 1)
predicted_label = class_labels[predicted_class.item()]  

print(f"Predicted class: {predicted_label}")


Predicted class: NRG
